In [ ]:
# Basis
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import contextily as cx
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import shutil

In [ ]:
# and from hydrolib-core
from hydrolib.core.dimr.models import DIMR, FMComponent
from hydrolib.core.dflowfm.inifield.models import IniFieldModel, DiskOnlyFileModel
from hydrolib.core.dflowfm.onedfield.models import OneDFieldModel
from hydrolib.core.dflowfm.structure.models import StructureModel
from hydrolib.core.dflowfm.crosssection.models import CrossLocModel, CrossDefModel
from hydrolib.core.dflowfm.ext.models import ExtModel
from hydrolib.core.dflowfm.mdu.models import FMModel
from hydrolib.core.dflowfm.friction.models import FrictionModel
from hydrolib.core.dflowfm.obs.models import ObservationPointModel
from hydrolib.core.dflowfm.storagenode.models import StorageNodeModel

In [ ]:
from hydrolib.dhydamo.core.hydamo import HyDAMO
from hydrolib.dhydamo.converters.df2hydrolibmodel import Df2HydrolibModel
from hydrolib.dhydamo.geometry import mesh
from hydrolib.dhydamo.core.drr import DRRModel
from hydrolib.dhydamo.core.drtc import DRTCModel
from hydrolib.dhydamo.io.dimrwriter import DIMRWriter
from hydrolib.dhydamo.io.drrwriter import DRRWriter
from hydrolib.dhydamo.geometry.viz import plot_network
from meshkernel.py_structures import DeleteMeshOption

In [ ]:
%load_ext autoreload
%autoreload 2

Define in- and output paths

In [ ]:
# selectie_gebied = 0 # Oude IJssel
selectie_gebied = 1 # West
# selectie_gebied = 2 # Centraal
# selectie_gebied = 3 # Oost

scenario = "scenario_test"

start_date = "2010-4-1"
end_date = "2010-4-11"

# path to the package containing the dummy-data
dir_data = Path("..\\..\\WRIJ_RR_Unpaved_methode_02_input\\rr_input_area_scenario")
dir_output = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel")
dir_assets = Path("..\\src\\wrij_rr_unpaved_methode\\assets")

In [ ]:
date_range = pd.date_range(start_date, end_date, freq="2D")
simulaties = pd.DataFrame(
    index=date_range,
    columns=["seizoen"],
    data=["zomer", "winter"]*int(len(date_range)/2)
).reset_index()
simulaties.columns = ["start_date", "seizoen"]

In [ ]:
simulaties

In [ ]:
# Hier begint de functie
def genereer_rr_model(dir_data, selectie_gebied, scenario, seizoen, start_date, end_date, restart_in):
    
    dir_gebied = Path(dir_data, f"gebied_{selectie_gebied}")
    dir_output_gebied = Path(dir_output, f"gebied_{selectie_gebied}")

    gebied = gpd.read_file(dir_gebied / f"gebied.gpkg", layer=f"gebied")
    watergang = gpd.read_file(dir_gebied / f"watergang.gpkg", layer=f"watergang")
    laterale_knoop = gpd.read_file(dir_gebied / f"laterale_knoop.gpkg", layer=f"laterale_knoop")
    afwateringseenheden = gpd.read_file(dir_gebied / f"afwateringseenheden.gpkg", layer=f"afwateringseenheden")

    #Initialize RR model
    drrmodel = DRRModel()

    #Read modelinput
    df_unpaved = pd.read_csv(dir_gebied / scenario / f"df_unpaved_{seizoen}.csv")
    df_ernst = pd.read_csv(dir_gebied / scenario / f"df_ernst_{seizoen}.csv")

    #add unpaved nodes
    #####Hier staat nu nog alleen zomer####
    for unpaved_dict in df_unpaved.set_index("code").to_dict("records"):
        drrmodel.unpaved.add_unpaved(**unpaved_dict)
    #add ernst definitions
    for ernst_dict in df_ernst.set_index("code").to_dict("records"):
        drrmodel.unpaved.add_ernst_def(**ernst_dict)

    # select project extend
    path_extent_project_area = dir_gebied / "gebied.gpkg"
    hydamo = HyDAMO(extent_file=path_extent_project_area)

    #read branches
    path_watergangen = Path(dir_gebied, "watergang.gpkg")
    hydamo.branches.read_gpkg_layer(path_watergangen, layer_name="watergang", index_col="code")

    # read catchments
    path_afwateringseenheden = Path(dir_gebied, "afwateringseenheden.gpkg")
    hydamo.catchments.read_gpkg_layer(path_afwateringseenheden, layer_name="afwateringseenheden", index_col="code", check_geotype=False)

    laterals_gdf = gpd.GeoDataFrame(df_unpaved, geometry=gpd.points_from_xy(df_unpaved.px-10, df_unpaved.py-10), crs=28992)
    laterals_gdf["rr_node"] = laterals_gdf["code"]
    laterals_gdf["code"] = laterals_gdf["boundary_node"]
    laterals_gdf["globalid"] = laterals_gdf["code"]
    laterals_gdf = laterals_gdf[["code", "rr_node", "globalid", "geometry"]].set_index("code")
    laterals_gdf.to_file(dir_data / "laterale_knoop.gpkg", layer="laterale_knoop", driver="GPKG")

    # read laterals and match them to the catchments
    hydamo.laterals.read_gpkg_layer(dir_data / "laterale_knoop.gpkg", layer_name="laterale_knoop")
    hydamo.laterals.snap_to_branch(hydamo.branches, snap_method="overal", maxdist=5000)
    hydamo.catchments['boundary_node'] = hydamo.catchments['lateraleknoopid'].copy()

    #generate RR boundaries
    drrmodel.external_forcings.io.boundary_from_input(
    hydamo.laterals, 
    hydamo.catchments, 
    drrmodel, 
    )

    #External forcings 
    #3 types of extrenal forcings need to be provided: seepage, precipitation and evaporation
    #read forcings
    seepage_file = str(dir_gebied / "seepage.csv")
    precip_file = str(dir_gebied / "meteo" / "METEO_NEERSLAG.BUI")
    evap_file = str(dir_gebied / "meteo" / "METEO_VERDAMPING.EVP")

    seepage_df = pd.read_csv(seepage_file, index_col=0, parse_dates=True)
    [drrmodel.external_forcings.add_seepage(*sep) for sep in seepage_df.items()]
    drrmodel.external_forcings.io.precip_from_input(hydamo.catchments, precip_folder=None, precip_file=precip_file)
    drrmodel.external_forcings.io.evap_from_input(hydamo.catchments, evap_folder=None, evap_file=evap_file)

    #Add the main parameters
    drrmodel.d3b_parameters["Timestepsize"] = 300
    drrmodel.d3b_parameters["StartTime"] = "'" + pd.to_datetime(start_date).strftime('%Y/%m/%d;%H:%M:%S') + "'"  # should be equal to refdate for D-HYDRO
    drrmodel.d3b_parameters["EndTime"] = "'" + pd.to_datetime(end_date).strftime('%Y/%m/%d;%H:%M:%S') + "'"
    drrmodel.d3b_parameters["RestartIn"] = 0
    drrmodel.d3b_parameters["RestartOut"] = 1
    drrmodel.d3b_parameters["RestartFileNamePrefix"] = "Test"
    drrmodel.d3b_parameters["UnsaturatedZone"] = 1
    drrmodel.d3b_parameters["UnpavedPercolationLikeSobek213"] = -1
    drrmodel.d3b_parameters["VolumeCheckFactorToCF"] = 100000

    #Convert laterals to RR boundaries
    hydamo.external_forcings.convert.laterals(
    hydamo.laterals,
    # overflows=hydamo.overflows,
    # greenhouse_laterals=hydamo.greenhouse_laterals,
    lateral_discharges=None,
        rr_boundaries=drrmodel.external_forcings.boundary_nodes
    )   

    #Writing the model
    # overwrite output-path to write the models
    output_path = dir_output_gebied / scenario / "rr_model"

    if not output_path.exists():
        output_path.mkdir(parents=True)

    rr_writer = DRRWriter(drrmodel, output_dir=output_path)
    rr_writer.write_all()

    #dir_assets = Path("..\\src\\wrij_rr_unpaved_methode\\assets")
    
    shutil.copy(
        dir_assets / "dimr_config.xml",
        output_path / "dimr_config.xml"
    )
    
    shutil.copy(
        dir_assets / "run.bat",
        output_path / "run.bat"
    )


    return drrmodel    

In [ ]:
for index, simulatie in simulaties.iterrows():
    print(simulatie.start_date)
    print(simulatie.seizoen)
    if index == 0:
        restart_in = 0
    else:
        restart_in = 1
    genereer_rr_model(dir_data, selectie_gebied, scenario, simulatie.seizoen, start_date, end_date, restart_in)